# 🧬 GAJE-Flow: Crianza y Destilación en la Nube (Google Colab)

Este cuaderno está configurado bajo el protocolo **GAJE-Flow v1.0.0-alpha** para compilar los kernels optimizados de Rust, montar los datasets de Google Drive e iniciar el proceso de destilación del modelo **Silver Adult** de forma segura a alto rendimiento.

## Paso 1: Configuración del Entorno de Ejecución
Asegúrate de cambiar el entorno de ejecución a una **GPU** o **CPU de alto rendimiento** (Entorno de ejecución -> Cambiar tipo de entorno de ejecución).

In [ ]:
# 1. Clonar el repositorio si no estás dentro de él
# !git clone https://github.com/tu-usuario/gaje-semantic-compression.git
# %cd gaje-semantic-compression

# 2. Instalar el compilador oficial de Rust y Maturin
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.cargo/bin")

!pip install maturin patchelf
!rustc --version

## Paso 2: Montar Google Drive para traer los datasets y modelos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copiar los archivos críticos de Google Drive al entorno local de Colab para lectura rápida
!mkdir -p data/datasets models/gguf models/core

print("[*] Copiando datasets...")
!cp /content/drive/MyDrive/GAJE_Backup/data/datasets/mosaic_dataset.txt data/datasets/

print("[*] Copiando modelos del profesor y configuraciones...")
!cp /content/drive/MyDrive/GAJE_Backup/models/gguf/smollm2-135m-f16.gguf models/gguf/
!cp /content/drive/MyDrive/GAJE_Backup/models/core/tokenizer.json models/core/

print("[✔] Archivos transferidos con éxito.")
!ls -lh data/datasets/ models/gguf/ models/core/

## Paso 3: Inicializar el organismo Silver Adult

In [ ]:
# Compilar gaje-cli e inicializar el modelo Silver de 10MB
!cargo run --release --bin gaje-cli -- --init models/silver_adult.gaje --preset silver_adult

## Paso 4: Compilación Nativa Optimizada y Ejecución del Entrenamiento

Compilamos con las banderas de optimización nativas para la arquitectura de la CPU del servidor de Google (Xeon/EPYC) para exprimir al máximo el rendimiento SIMD.

In [ ]:
# Compilar con optimizaciones nativas de CPU (AVX-512 / AVX2)
os.environ["RUSTFLAGS"] = "-C target-cpu=native"
!cargo build --release --bin micro-distiller

# Lanzar el proceso de destilación equilibrada para 3 épocas
!./target/release/micro-distiller --student models/silver_adult.gaje --output models/silver_adult.gaje --dataset data/datasets/mosaic_dataset.txt --epochs 3

## Paso 5: Respaldar el modelo entrenado de vuelta a tu Google Drive

In [ ]:
# Respaldar el modelo entrenado y listo para usar en tu Drive
!cp models/silver_adult.gaje /content/drive/MyDrive/GAJE_Backup/models/
print("[✔] Modelo respaldado en Google Drive. Ya puedes descargarlo para usarlo en Android o localmente.")